# 03 · ETL de movimiento

Agrega por **streaming** (lectura por chunks) los ficheros de sensor `_F` (fragmentos) y `_T` (por hora)
de cada participante — en total ~2,7 GB que **no** se cargan enteros en memoria.

Genera tres JSON ligeros en `docs/data/`:
- **`daily_summary.json`** — participante × día (intensidad, fracción activa, velocidad).
- **`hourly_activity.json`** — curva circadiana (24 medias de intensidad) por participante + media global.
- **`derived_features.json`** — features agregadas (1 fila/participante) para correlación y clustering.

La señal base de movimiento es la magnitud de la aceleración de usuario
`sqrt(uacc_x² + uacc_y² + uacc_z²)` en G (robusta frente al pedómetro).

In [1]:
import sys, os, time
sys.path.append(os.path.abspath('../src'))
import etl_utils as eu
import pandas as pd

inv = eu.list_sensor_files()
print('Participantes con ficheros de sensor:', len(inv))

Participantes con ficheros de sensor: 58


In [2]:
daily_all, hourly_all, derived_all = [], [], []
t0 = time.time()
for i, (pid, files) in enumerate(inv.items(), 1):
    daily, hourly, derived = eu.aggregate_participant_movement(pid, files)
    if derived is None:
        continue
    daily_all.extend(daily)
    hourly_all.append(hourly)
    derived_all.append(derived)
    print(f'[{i:2d}/{len(inv)}] {pid:5s} dias={len(daily):2d} n={derived["n_samples"]:>9,} '
          f'({time.time()-t0:5.1f}s)')
print('\nTotal:', len(derived_all), 'participantes |', len(daily_all), 'dias |',
      f'{time.time()-t0:.1f}s')

[ 1/58] B26   dias= 1 n=    6,751 (  0.1s)
[ 2/58] C10   dias= 2 n=  113,347 (  1.6s)
[ 3/58] C28   dias= 2 n=  110,291 (  3.0s)
[ 4/58] C41   dias= 1 n=    6,510 (  3.1s)
[ 5/58] C42   dias= 1 n=    6,610 (  3.2s)
[ 6/58] C43   dias= 2 n=  113,168 (  4.6s)
[ 7/58] F55   dias= 1 n=   73,643 (  5.6s)
[ 8/58] G40   dias= 1 n=    7,933 (  5.7s)
[ 9/58] H1    dias= 1 n=    6,646 (  5.8s)
[10/58] H2    dias= 2 n=  113,841 (  7.2s)
[11/58] H44   dias= 2 n=  113,068 (  8.7s)
[12/58] H45   dias= 2 n=  113,236 ( 10.2s)
[13/58] H46   dias= 2 n=   83,205 ( 11.3s)
[14/58] H47   dias= 2 n=  113,216 ( 12.8s)
[15/58] H48   dias= 2 n=  113,231 ( 14.2s)
[16/58] H49   dias= 1 n=    6,959 ( 14.3s)
[17/58] H50   dias= 2 n=  113,173 ( 15.8s)
[18/58] J33   dias= 1 n=    6,561 ( 15.8s)
[19/58] J34   dias= 2 n=  113,332 ( 17.3s)
[20/58] L15   dias= 1 n=    6,553 ( 17.4s)
[21/58] L16   dias= 2 n=  113,572 ( 18.9s)
[22/58] L17   dias= 1 n=    6,608 ( 19.0s)
[23/58] L18   dias= 1 n=    6,664 ( 19.1s)
[24/58] L29

## Curva circadiana media global

Como cada niño solo graba unas horas, promediamos la intensidad por hora **entre participantes**
para obtener un patrón circadiano poblacional más completo.

In [3]:
import numpy as np
global_hours = []
for h in range(24):
    vals = [rec['hours'][h]['mean_intensity'] for rec in hourly_all
            if rec['hours'][h]['mean_intensity'] is not None]
    global_hours.append({
        'hour': h,
        'phase': eu._circadian_phase(h),
        'mean_intensity': (float(np.mean(vals)) if vals else None),
        'n_participants': len(vals),
    })
hourly_payload = {'global': global_hours, 'by_participant': hourly_all}
pd.DataFrame(global_hours)

,hour,phase,mean_intensity,n_participants
0,0,night,NaN,0
1,1,night,NaN,0
2,2,night,NaN,0
3,3,night,NaN,0
4,4,night,NaN,0
5,5,morning,NaN,0
6,6,morning,NaN,0
7,7,morning,NaN,0
8,8,morning,0.217357,2
9,9,morning,0.553798,5


## Exportar JSON

In [4]:
eu.write_json(daily_all, 'daily_summary.json')
eu.write_json(hourly_payload, 'hourly_activity.json')
eu.write_json(derived_all, 'derived_features.json')
print('Hecho.')
pd.DataFrame(derived_all).describe().round(3)

  escrito daily_summary.json  (12.8 KB)
  escrito hourly_activity.json  (93.0 KB)
  escrito derived_features.json  (15.8 KB)
Hecho.


,n_samples,n_days,mean_intensity,accel_variance,active_fraction,sedentary_fraction,activity_fragmentation,weekend_delta,circadian_stability,peak_hour,mobility_radius_m
count,58.000,58.000,58.000,58.000,58.000,58.000,58.000,4.000,58.000,58.000,58.000
mean,64772.810,1.431,0.556,0.292,0.832,0.168,0.034,-0.199,0.941,16.517,90.289
std,51368.444,0.500,0.169,0.185,0.118,0.118,0.016,0.283,0.069,1.128,63.397
min,6510.000,1.000,0.257,0.055,0.506,0.028,0.006,-0.531,0.740,11.000,8.438
25%,6746.500,1.000,0.447,0.134,0.784,0.062,0.018,-0.339,0.894,16.000,39.353
50%,103849.000,1.000,0.533,0.259,0.858,0.142,0.036,-0.206,0.969,17.000,79.520
75%,113158.250,2.000,0.640,0.431,0.938,0.216,0.044,-0.066,1.000,17.000,124.694
max,113841.000,2.000,1.295,0.813,0.972,0.494,0.071,0.147,1.000,18.000,252.586
